In [1]:
import pandas as pd
import os
import glob
import numpy as np

# Paths
input_folder = "data"
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

def get_season(month):
    if month in [12, 1, 2]: return 'summer'
    if month in [3, 4, 5]: return 'fall'
    if month in [6, 7, 8]: return 'winter'
    if month in [9, 10, 11]: return 'spring'

summary_results = []
hourly_profiles = []
thermal_results = []
water_heating_summaries = []
all_power_profiles = {} 

for filepath in glob.glob(os.path.join(input_folder, "*.csv")):
    municipality = os.path.basename(filepath).replace("ninja_weather_", "").replace(".csv", "")
    
    df = pd.read_csv(filepath, comment='#')
    df['local_time'] = pd.to_datetime(df['local_time'])
    df['month'] = df['local_time'].dt.month
    df['hour'] = df['local_time'].dt.hour
    df['date'] = df['local_time'].dt.date
    df['season'] = df['month'].apply(get_season)

    # Daily Aggregation
    daily_stats = df.groupby(['date', 'season'])['t2m'].agg(['mean', 'max', 'min']).reset_index()
    t_annual_mean = df['t2m'].mean()
    
    # Water Heating Power Calculation
    A_gw = 1.0
    phase_lag = 30
    flow_rate = 0.08
    cp_water = 4.18
    target_temp = 40

    daily_stats['day_of_year'] = pd.to_datetime(daily_stats['date']).dt.dayofyear
    daily_stats['T_gw'] = t_annual_mean + A_gw * np.cos(2 * np.pi / 365 * (daily_stats['day_of_year'] - phase_lag))
    daily_stats['power_W'] = flow_rate * cp_water * (target_temp - daily_stats['T_gw']) * 1000
    
    # FIX: Group by day_of_year to handle duplicates (e.g., leap years or data overlaps)
    clean_daily_power = daily_stats.groupby('day_of_year')['power_W'].mean()
    all_power_profiles[municipality] = clean_daily_power

    # Water Heating Summary
    water_heating_summaries.append({
        'municipality': municipality,
        'annual_mean_power_W': daily_stats['power_W'].mean(),
        'winter_mean_power_W': daily_stats[daily_stats['season'] == 'winter']['power_W'].mean(),
        'summer_mean_power_W': daily_stats[daily_stats['season'] == 'summer']['power_W'].mean(),
        'max_power_W': daily_stats['power_W'].max(),
        'min_power_W': daily_stats['power_W'].min()
    })

    # Summary and Thermal Comfort
    summer_max_mean = daily_stats[daily_stats['season'] == 'summer']['max'].mean()
    winter_min_mean = daily_stats[daily_stats['season'] == 'winter']['min'].mean()

    summary_results.append({
        'municipality': municipality,
        'annual_mean_temp': t_annual_mean,
        'summer_mean_temp': daily_stats[daily_stats['season'] == 'summer']['mean'].mean(),
        'fall_mean_temp': daily_stats[daily_stats['season'] == 'fall']['mean'].mean(),
        'winter_mean_temp': daily_stats[daily_stats['season'] == 'winter']['mean'].mean(),
        'spring_mean_temp': daily_stats[daily_stats['season'] == 'spring']['mean'].mean(),
        'summer_max_temp': summer_max_mean,
        'winter_min_temp': winter_min_mean,
        'annual_min_hour': df['t2m'].min(),
        'annual_max_hour': df['t2m'].max()
    })

    profile = df.groupby(['season', 'hour'])['t2m'].mean().reset_index()
    profile['municipality'] = municipality
    hourly_profiles.append(profile)

    df['is_hot'] = df['t2m'] > 26
    daily_hot_counts = df.groupby(['date', 'season'])['is_hot'].sum().reset_index()
    seasonal_hot_means = daily_hot_counts.groupby('season')['is_hot'].mean().reset_index()
    for _, row in seasonal_hot_means.iterrows():
        func_time = max(60, min(600, round((row['is_hot'] * 60) / 30) * 30))
        thermal_results.append({
            'municipality': municipality, 'season': row['season'],
            'mean_daily_hours_above_26': row['is_hot'], 'func_time': int(func_time)
        })

# Save Outputs
df_power_all = pd.DataFrame(all_power_profiles).sort_index()
df_power_all.index.name = 'day'
df_power_all.reset_index().to_csv(os.path.join(output_folder, "water_heating_power_all_municipalities.csv"), index=False)

pd.DataFrame(water_heating_summaries).to_csv(os.path.join(output_folder, "water_heating_summary.csv"), index=False)
pd.DataFrame(summary_results).to_csv(os.path.join(output_folder, "municipality_temperatures.csv"), index=False)
pd.DataFrame(thermal_results).to_csv(os.path.join(output_folder, "thermal_comfort_lookup.csv"), index=False)

df_profiles = pd.concat(hourly_profiles)
df_pivot = df_profiles.pivot(index=['municipality', 'season'], columns='hour', values='t2m').reset_index()
df_pivot.columns = ['municipality', 'season'] + [f'hour_{i}' for i in range(24)]
df_pivot.to_csv(os.path.join(output_folder, "hourly_profiles_by_season.csv"), index=False)

print("Done! All files generated successfully.")

Done! All files generated successfully.


In [2]:
h8= (df_pivot['hour_8'].sum())/21/4
h21= (df_pivot['hour_21'].sum())/21/4
h22= (df_pivot['hour_22'].sum())/21/4
print(f'at 8am = {h8}, at 9pm = {h21}, at 10pm = {h22}')

at 8am = 24.898121509023525, at 9pm = 25.23846936986984, at 10pm = 24.806712153670464
